# GRU-CVAE

- Model for extracted time-series data from KineMatrix
- Time-series model with hold-out-set

*Note:* Cannot be trained if NaN values are existing

- Load packages

    - There can be some problems with the tensorflow, make sure to install it, so it fits with your specific computer. 

        - If you get an error with JAX / jaxlib use the command:  *pip uninstall -y jax jaxlib* in your conda environment. This code will work without jaxlib.

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
import pandas as pd
import numpy as np
print("NumPy:", np.__version__)
from tensorflow import keras
from tensorflow.keras import layers, Model, metrics
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K
from tensorflow.keras import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from typing import List, Tuple
import matplotlib.pyplot as plt
import plotly.express as px
import os

- Define helper function
    - NormPhase_fore is now your explicit time axis
    - Each sequence is sorted according to the loop

In [ ]:
# ==============================================================================
#  Data Processing Functions
# ==============================================================================

def prepare_and_scale_data(filepath, timesteps, test_path):
    """Loads, splits by MouseID, strictly scales, and reshapes into 3D tensors."""
    df = pd.read_csv(filepath)
    
    # 1. Split Data by MouseID: Create the Final Test Set (formerly Holdout)
    # 15% of the TOTAL data is saved for the final Test set
    gss_final_test = GroupShuffleSplit(n_splits=1, test_size=0.14, random_state=42) 
    main_idx, test_idx = next(gss_final_test.split(df, groups=df['MouseID']))
    
    df_main = df.iloc[main_idx].copy()
    df_test = df.iloc[test_idx].copy()
    
    # 2. Split Main Data: Create the Validation Set
        # We want Validation to be 15% of the ORIGINAL total.
        # We currently have 85% of the data in df_main.
        # 15 / 85 = 0.17647...
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.16, random_state=42)
    train_idx, val_idx = next(gss_val.split(df_main, groups=df_main['MouseID']))
    
    df_train = df_main.iloc[train_idx].copy()
    df_val = df_main.iloc[val_idx].copy()

    # --- Print split checks ---
    total_len = len(df)
    print(f"Split Ratios: Train={len(df_train)/total_len:.2f}, Val={len(df_val)/total_len:.2f}, Test={len(df_test)/total_len:.2f}")
    print("\nNote: this may not be exactly 70%,15%,15%, because we split by MouseID, so we dont split mice in different groups. ")

    # 3. Identify Features vs Metadata
    grouping_cols = ['MouseID', 'Side', 'Run', 'cycle_id_fore']
    metadata_cols = ['Treatment', 'State', 'Person', 'Injury_Label', 'Frame']
    cols_to_exclude = grouping_cols + metadata_cols + ['NormPhase_fore']
    
    feature_cols = [c for c in df_train.columns if c not in cols_to_exclude and df_train[c].dtype in ['float64', 'int64']]
    n_features = len(feature_cols)

    # 4. Strict Scaling (FIT ONLY ON TRAIN)
    scaler = StandardScaler()
    scaler.fit(df_train[feature_cols])
    
    df_train[feature_cols] = scaler.transform(df_train[feature_cols])
    df_val[feature_cols] = scaler.transform(df_val[feature_cols])    # Scale Validation
    df_test[feature_cols] = scaler.transform(df_test[feature_cols])  # Scale Test
    
    # Save Final Test Set
    df_test.to_csv(test_path, index=False)

    # 5. Reshape Function (Flat to 3D)
    def reshape_to_3d(subset_df):
        data = subset_df[feature_cols].values
        labels = subset_df['Injury_Label'].values.astype(int)
        
        n_samples = len(subset_df) // timesteps
        rows_needed = n_samples * timesteps
        
        data = data[:rows_needed]
        labels = labels[:rows_needed]
        meta_sliced = subset_df.iloc[:rows_needed].copy()
        
        data_reshaped = data.reshape(n_samples, timesteps, n_features)
        labels_reshaped = labels[::timesteps]
        meta_reshaped = meta_sliced.iloc[::timesteps][grouping_cols + metadata_cols].reset_index(drop=True)
        
        return data_reshaped, labels_reshaped, meta_reshaped

    X_train, y_train, meta_train = reshape_to_3d(df_train)
    X_val, y_val, meta_val = reshape_to_3d(df_val)     
    X_test, y_test, meta_test = reshape_to_3d(df_test) 

    return X_train, y_train, meta_train, X_val, y_val, meta_val, X_test, y_test, meta_test, n_features

def balance_single_dataset(data, labels, meta, dataset_name="Train"):
    """
    Balances 0 and 1 classes for a specific set. 
    DISCARDS excess data (does not move to Unlabelled).
    """
    meta = meta.copy()
    
    # Identify indices
    idx_unlabelled = np.where(labels == -1)[0]
    idx_healthy = np.where(labels == 0)[0]
    idx_severe = np.where(labels == 1)[0]
    
    # Find minimum count to equalize
    min_samples = min(len(idx_healthy), len(idx_severe))
    rng = np.random.default_rng(42)

    # Select balanced samples
    idx_healthy_balanced = rng.choice(idx_healthy, size=min_samples, replace=False)
    idx_severe_balanced = rng.choice(idx_severe, size=min_samples, replace=False)
    
    # Assign Plot Groups
    meta.loc[idx_healthy_balanced, 'Plot_Group'] = f'Healthy - {dataset_name}'
    meta.loc[idx_severe_balanced, 'Plot_Group'] = f'Severe - {dataset_name}'
    if len(idx_unlabelled) > 0:
        meta.loc[idx_unlabelled, 'Plot_Group'] = f'Unlabelled - {dataset_name}'

    # Combine indices: Only Unlabelled(original) + Balanced Healthy + Balanced Severe
    # Excess indices are simply ignored/dropped here
    final_indices = np.concatenate([idx_unlabelled, idx_healthy_balanced, idx_severe_balanced])
    
    # Grab corresponding labels
    final_labels = labels[final_indices]
    
    # Shuffle the result
    shuffle_order = rng.permutation(len(final_indices))
    final_indices = final_indices[shuffle_order]
    final_labels = final_labels[shuffle_order]
    
    new_data = data[final_indices]
    new_meta = meta.iloc[final_indices].reset_index(drop=True)
    
    print(f"{dataset_name} set balanced: {min_samples} per class (steps).")
    return new_data, final_labels, new_meta

def assign_test_set_groups(data, labels, meta, dataset_name="Test"):
    """
    Assigns Plot_Group labels to the Test set without balancing or dropping data.
    This preserves the natural distribution for unbiased evaluation.
    """
    meta = meta.copy()
    
    # Create masks based on labels
    mask_healthy = (labels == 0)
    mask_severe = (labels == 1)
    # Any label not 0 or 1 is considered Unlabelled (usually -1)
    mask_unlabelled = ~(mask_healthy | mask_severe) 
    
    # Assign names
    meta.loc[mask_healthy, 'Plot_Group'] = f'Healthy - {dataset_name}'
    meta.loc[mask_severe, 'Plot_Group'] = f'Severe - {dataset_name}'
    meta.loc[mask_unlabelled, 'Plot_Group'] = f'Unlabelled - {dataset_name}'
    
    print(f"Assigned plot groups to {dataset_name} set (n={len(data)}).")
    return data, labels, meta

def print_unique_mice_per_split(y, meta, split_name):
    # Create a temporary dataframe linking labels to Mouse IDs
    temp_df = pd.DataFrame({'Label': y, 'MouseID': meta['MouseID']})
    
    # Map the numerical labels to readable names
    label_map = {-1: 'Unlabelled', 0: 'Healthy', 1: 'Severe'}
    temp_df['Group'] = temp_df['Label'].map(label_map)
    
    # Count unique MouseIDs per group
    counts = temp_df.groupby('Group')['MouseID'].nunique()
    total_mice = temp_df['MouseID'].nunique()
    
    print(f"{split_name} Set -> Total Unique Mice: {total_mice}")
    for group, count in counts.items():
        print(f"    - {group}: {count} mice")

- Define GRU-CVAE Arcitecture components

In [ ]:
# 1. Custom Sampling Layer

class Sampling(layers.Layer):
    """
    Reparameterization for sampling from the latent space. 
    (z = mu + epsilon * std).
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        timesteps = tf.shape(z_mean)[1]
        dim = tf.shape(z_mean)[2]
        # sample 'epsilon' from a normal distribution
        epsilon = tf.keras.backend.random_normal(shape=(batch, timesteps, dim))
        # calculate z
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


# 2. Model Builders (Encoder, Decoder, Classifier)
    
def build_encoder(timesteps, n_features, latent_dim):
    """
    Build the encoder (input for the latent space).
    Conv1D is used in this model.

    """
    # input layer
    inputs = layers.Input(shape=(timesteps, n_features), name="encoder_input")
    
    # convolutional layers to catch temporal patterns
    x = layers.Conv1D(32, 5, activation="relu", padding="same")(inputs)
    x = layers.Conv1D(64, 5, activation="relu", padding="same")(x)
    
    # temporal Dependency (trajectory)
        # return sequences = True to keep the 100 timesteps
    x = layers.Bidirectional(layers.GRU(32, return_sequences=True))(x)
    
    # output mu (mean) og log_var (log-varians)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    
    # reparameterization
    z = Sampling()([z_mean, z_log_var])
    
    return Model(inputs, [z, z_mean, z_log_var], name="encoder")


def build_decoder(timesteps, n_features, latent_dim, num_classes=2):
    """
    Decoder now takes (latent_dim + num_classes) as input channels.
    """
    # Adjust input shape to accommodate concatenated One-Hot labels
    input_dim = latent_dim + num_classes
    latent_inputs = layers.Input(shape=(timesteps, input_dim), name="decoder_input")
    
    x = layers.GRU(64, return_sequences=True)(latent_inputs)
    x = layers.Conv1D(32, 5, activation="relu", padding="same")(x)
    decoder_outputs = layers.Conv1D(n_features, 5, activation="linear", padding="same")(x)
    
    return Model(latent_inputs, decoder_outputs, name="decoder")


def build_classifier(timesteps, latent_dim, num_classes):
    """
    Classifies the sequence of latent vectors.
    Input: (None, 100, 2) -> Output: (None, num_classes)
    """
    inputs = layers.Input(shape=(timesteps, latent_dim), name="clf_input")
    
    # Flatten the trajectory sequence into a single feature vector
    x = layers.Flatten()(inputs)
    
    # Classification Head
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="clf_output")(x) # final decision
    
    return Model(inputs, outputs, name="classifier")


- Make the GRU-CVAE model

In [ ]:
def repeat_and_concat_label(x_seq, y_vec):
    """
    x_seq: (Batch, Timesteps, Latent)
    y_vec: (Batch, Classes) - One Hot
    returns: (Batch, Timesteps, Latent + Classes)
    """
    T = tf.shape(x_seq)[1]
    # Expand y_vec to (Batch, 1, Classes) and tile it to match timesteps
    y_rep = tf.tile(tf.expand_dims(y_vec, axis=1), [1, T, 1]) 
    return tf.concat([x_seq, y_rep], axis=-1)

class SemiSupervised_GRU_CVAE(keras.Model):
    def __init__(self, encoder, decoder, classifier, alpha=1.0, num_classes=2, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.classifier = classifier
        self.alpha = alpha
        self.num_classes = num_classes
        
        # Trackers
        self.total_loss_tracker = metrics.Mean(name="total_loss")
        self.recon_loss_tracker = metrics.Mean(name="recon_loss")
        self.kl_loss_tracker = metrics.Mean(name="kl_loss")
        self.class_loss_tracker = metrics.Mean(name="class_loss")
        self.acc_tracker = metrics.Accuracy(name="clf_accuracy")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.recon_loss_tracker,
            self.kl_loss_tracker,
            self.class_loss_tracker,
            self.acc_tracker
        ]

    def train_step(self, data):
        if isinstance(data, tuple):
            data_x = data[0]
            data_y = data[1] # Labels: -1, 0, 1
        else:
            data_x = data
            data_y = None

        with tf.GradientTape() as tape:
            # 1. Encoder Forward Pass
            z, z_mean, z_log_var = self.encoder(data_x)
            
            # 2. Classifier Forward Pass (Get predictions for ALL data)
            class_pred = self.classifier(z) # Softmax output (Batch, 2)
            
            # ---------------------------------------------------------
            # 3. PREPARE CONDITION (One-Hot Logic)
            # ---------------------------------------------------------
            # Create a mask: 1 where labeled, 0 where -1 (unlabeled)
            is_labeled = tf.cast(tf.not_equal(data_y, -1), tf.float32)
            is_labeled_vec = tf.expand_dims(is_labeled, 1) # Shape (Batch, 1)

            # A. Prepare Ground Truth One-Hot
            # Clip -1 to 0 temporarily so one_hot function doesn't crash
            safe_labels = tf.maximum(data_y, 0)
            y_onehot_true = tf.one_hot(tf.cast(safe_labels, tf.int32), depth=self.num_classes)
            
            # B. Combine: If labeled -> Use True Label. If unlabeled -> Use Prediction.
            # This allows the Decoder to reconstruct unlabeled data using the inferred class.
            y_condition = (is_labeled_vec * y_onehot_true) + ((1 - is_labeled_vec) * class_pred)
            
            # C. Concatenate z + y_condition
            z_conditional = repeat_and_concat_label(z, y_condition)
            
            # ---------------------------------------------------------

            # 4. Decoder Forward Pass (using Conditional Z)
            reconstruction = self.decoder(z_conditional)
            
            # 5. Losses
            # Reconstruction (MSE)
            recon_loss = tf.reduce_mean(tf.reduce_mean(tf.square(data_x - reconstruction), axis=(1, 2)))

            # KL Divergence
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_mean(kl_loss, axis=(1, 2)))
            
            # Classification Loss (Supervised - Masked for labeled only)
            cce = keras.losses.SparseCategoricalCrossentropy(reduction='none')
            class_loss_raw = cce(safe_labels, class_pred)
            class_loss_masked = class_loss_raw * is_labeled # Zero out unlabeled
            
            # Average over only the number of labeled samples
            num_labeled = tf.reduce_sum(is_labeled)
            class_loss = tf.math.divide_no_nan(tf.reduce_sum(class_loss_masked), num_labeled)
            
            # Total Loss
            total_loss = recon_loss + kl_loss + (self.alpha * class_loss)

        # Backpropagation
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # Metrics
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        self.class_loss_tracker.update_state(class_loss)
        
        pred_labels = tf.argmax(class_pred, axis=1, output_type=tf.int32)
        self.acc_tracker.update_state(safe_labels, pred_labels, sample_weight=is_labeled)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
            "class_loss": self.class_loss_tracker.result(),
            "accuracy": self.acc_tracker.result(),
        }

    def test_step(self, data):
        if isinstance(data, tuple):
            data_x, data_y = data
        else:
            data_x, data_y = data, None
            
        z, z_mean, z_log_var = self.encoder(data_x)
        class_pred = self.classifier(z)
        
        # Logic for test step conditioning (same as train)
        is_labeled = tf.cast(tf.not_equal(data_y, -1), tf.float32)
        is_labeled_vec = tf.expand_dims(is_labeled, 1)
        safe_labels = tf.maximum(data_y, 0)
        y_onehot_true = tf.one_hot(tf.cast(safe_labels, tf.int32), depth=self.num_classes)
        
        # In testing, we can often just use the prediction for everything if we want,
        # but to evaluate loss correctly, we use the same logic:
        y_condition = (is_labeled_vec * y_onehot_true) + ((1 - is_labeled_vec) * class_pred)
        
        z_conditional = repeat_and_concat_label(z, y_condition)
        reconstruction = self.decoder(z_conditional)
        
        recon_loss = tf.reduce_mean(tf.reduce_mean(tf.square(data_x - reconstruction), axis=(1, 2)))
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_mean(kl_loss, axis=(1, 2)))
        
        cce = keras.losses.SparseCategoricalCrossentropy(reduction='none')
        class_loss = tf.math.divide_no_nan(
            tf.reduce_sum(cce(safe_labels, class_pred) * is_labeled), 
            tf.reduce_sum(is_labeled)
        )
        
        total_loss = recon_loss + kl_loss + (self.alpha * class_loss)
        
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        self.class_loss_tracker.update_state(class_loss)
        pred_labels = tf.argmax(class_pred, axis=1, output_type=tf.int32)
        self.acc_tracker.update_state(safe_labels, pred_labels, sample_weight=is_labeled)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "recon": self.recon_loss_tracker.result(),
            "kl": self.kl_loss_tracker.result(),
            "class": self.class_loss_tracker.result(),
            "acc": self.acc_tracker.result(),
        }

- Define model parameters

In [ ]:
############################################################################################
################################### Model parameters #######################################

timesteps = 100      # T=100
latent_dim = 16
test_size = 0.2      # train/test = 80/20
random_state = 23    # for reproducability
batch_size = 128      # number of datapoints passed through the model in sets during training
epochs = 10000       
alpha_classification_weight = 1.0 # Higher alpha forces better separation
learning_rate = 1e-5 # learning rate for Adam optimizer; 0.0001
                        # - too low the loss function doesnt improve
                        # - too high the curve begins to diverge + spike up and down
                            # 0.0005 was used for the dim=2
num_classes = 2


# set the path for your data file to be analysed:
filepath = "/Users/asm/Desktop/UNI/MasterThesis/data/final/gru_data/preprocessed_datafile_gru_nonscaled.csv" 
output_dir = "/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/"

holdout_path = "/Users/asm/Desktop/UNI/MasterThesis/data/final/gru_data/slet_holdout_set_scaled.csv"


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============================================================
# SETTINGS YOU PROVIDED
# ============================================================
test_path = "/Users/asm/Desktop/UNI/MasterThesis/data/final/slet.csv"
os.makedirs(os.path.dirname(test_path), exist_ok=True)
print(f"[Path] Holdout test will be saved to: {test_path}")

# ============================================================
# Cross-validation data preparation (NO scaling leakage)
# ============================================================
def prepare_cross_validation(filepath, timesteps, test_path, n_splits=10, random_state=42):
    print("\n==============================")
    print(" Preparing data for Cross-Validation")
    print("==============================")

    df = pd.read_csv(filepath)
    print(f"[Load] Loaded dataframe: {df.shape} (rows, cols)")

    # 1) Final holdout test split by MouseID
    gss_final_test = GroupShuffleSplit(n_splits=1, test_size=0.14, random_state=random_state)
    main_idx, test_idx = next(gss_final_test.split(df, groups=df["MouseID"]))

    df_main = df.iloc[main_idx].copy()
    df_test = df.iloc[test_idx].copy()

    total_len = len(df)
    print(f"[Split] Main pool rows: {len(df_main)} ({len(df_main)/total_len:.2f})")
    print(f"[Split] Holdout test rows: {len(df_test)} ({len(df_test)/total_len:.2f})")
    print("Note: split is by MouseID to avoid mice leaking across splits.")

    grouping_cols = ["MouseID", "Side", "Run", "cycle_id_fore"]
    metadata_cols = ["Treatment", "State", "Person", "Injury_Label", "Frame"]
    cols_to_exclude = grouping_cols + metadata_cols + ["NormPhase_fore"]

    feature_cols = [
        c for c in df_main.columns
        if c not in cols_to_exclude and df_main[c].dtype in ["float64", "int64"]
    ]
    n_features = len(feature_cols)
    print(f"[Features] Using {n_features} numeric feature columns")

    def reshape_to_3d_raw(subset_df, subset_name="MAIN"):
        data = subset_df[feature_cols].values
        labels = subset_df["Injury_Label"].values.astype(int)

        n_samples = len(subset_df) // timesteps
        rows_needed = n_samples * timesteps

        data = data[:rows_needed]
        labels = labels[:rows_needed]
        meta_sliced = subset_df.iloc[:rows_needed].copy()

        X_3d = data.reshape(n_samples, timesteps, n_features)
        y_seq = labels[::timesteps]

        meta_seq = meta_sliced.iloc[::timesteps][grouping_cols + metadata_cols].reset_index(drop=True)
        groups_seq = meta_seq["MouseID"].values

        print(f"[Reshape-{subset_name}] X: {X_3d.shape}, y: {y_seq.shape}, unique mice: {pd.Series(groups_seq).nunique()}")
        return X_3d, y_seq, meta_seq, groups_seq

    X_cv_raw, y_cv, meta_cv, groups_cv = reshape_to_3d_raw(df_main, subset_name="CV_POOL")

    # Save holdout CSV (since you gave a real path)
    df_test.to_csv(test_path, index=False)
    print(f"[Save] Holdout test CSV saved to: {test_path}")

    print("\nDone preparing CV pool (UNSCALED). Scaling will happen inside each fold.")
    return X_cv_raw, y_cv, meta_cv, groups_cv, df_test, n_features, feature_cols


# ============================================================
# StratifiedGroupKFold CV + per-fold scaling + ONE combined plot
# ============================================================
print("\n==============================")
print(" Running StratifiedGroupKFold CV (with per-fold scaling)")
print("==============================")

# NOTE: you must already have these variables from your notebook:
# filepath, timesteps, latent_dim, num_classes, alpha_classification_weight,
# learning_rate, epochs, batch_size
# and these functions/classes:
# balance_single_dataset, build_encoder, build_decoder, build_classifier, SemiSupervised_GRU_CVAE

X_cv_raw, y_cv, meta_cv, groups_cv, df_test_raw, n_features, feature_cols = prepare_cross_validation(
    filepath=filepath,
    timesteps=timesteps,
    test_path=test_path,
    n_splits=10,
    random_state=42
)

# Treat -1 as a separate class for stratification stability
y_strat = np.where(y_cv == -1, 2, y_cv)

cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

fold_histories = []

def _dist(y):
    u, c = np.unique(y, return_counts=True)
    return dict(zip(u.tolist(), c.tolist()))

for fold, (train_idx, val_idx) in enumerate(cv.split(X_cv_raw, y_strat, groups=groups_cv), start=1):
    print("\n------------------------------------------")
    print(f" Fold {fold}/10")
    print("------------------------------------------")

    # RAW splits
    X_tr_raw, y_tr = X_cv_raw[train_idx], y_cv[train_idx]
    X_va_raw, y_va = X_cv_raw[val_idx], y_cv[val_idx]

    meta_tr = meta_cv.iloc[train_idx].reset_index(drop=True)
    meta_va = meta_cv.iloc[val_idx].reset_index(drop=True)

    print(f"[Raw] Train X: {X_tr_raw.shape} | Val X: {X_va_raw.shape}")
    print(f"[Raw] Train label dist: {_dist(y_tr)}")
    print(f"[Raw] Val   label dist: {_dist(y_va)}")
    print(f"[Groups] Unique mice (train): {pd.Series(meta_tr['MouseID']).nunique()} | (val): {pd.Series(meta_va['MouseID']).nunique()}")

    # Per-fold scaling (NO leakage)
    print("[Scale] Fitting scaler on TRAIN fold only (no leakage).")
    scaler = StandardScaler()
    scaler.fit(X_tr_raw.reshape(-1, n_features))

    X_tr = scaler.transform(X_tr_raw.reshape(-1, n_features)).reshape(X_tr_raw.shape)
    X_va = scaler.transform(X_va_raw.reshape(-1, n_features)).reshape(X_va_raw.shape)
    print("[Scale] Done transforming train + val using fold scaler.")

    # Balance inside fold (your function)
    print("[Balance] Balancing train/val inside this fold...")
    X_tr_bal, y_tr_bal, meta_tr_bal = balance_single_dataset(X_tr, y_tr, meta_tr, dataset_name=f"Fold{fold}-Train")
    X_va_bal, y_va_bal, meta_va_bal = balance_single_dataset(X_va, y_va, meta_va, dataset_name=f"Fold{fold}-Val")

    print(f"[Balanced] Train label dist: {_dist(y_tr_bal)}")
    print(f"[Balanced] Val   label dist: {_dist(y_va_bal)}")
    print(f"[Balanced] Train X: {X_tr_bal.shape} | Val X: {X_va_bal.shape}")

    # Fresh model per fold
    print("[Model] Building fresh model for this fold...")
    encoder = build_encoder(timesteps, n_features, latent_dim)
    decoder = build_decoder(timesteps, n_features, latent_dim, num_classes)
    classifier = build_classifier(timesteps, latent_dim, num_classes)

    ss_vae = SemiSupervised_GRU_CVAE(
        encoder, decoder, classifier,
        alpha=alpha_classification_weight,
        num_classes=num_classes
    )
    ss_vae.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))

    early_stopping = EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-7, verbose=1)

    print("[Fit] Training...")
    history = ss_vae.fit(
        X_tr_bal, y_tr_bal,
        validation_data=(X_va_bal, y_va_bal),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    print("[Fit] Done.")

    fold_histories.append(history.history)
    print(f"[Fold {fold}] Finished. Epochs run: {len(history.history.get('loss', []))}")


# ============================================================
# ONE combined plot: mean ± std across folds (train + val)
# ============================================================
print("\n==============================")
print(" Plotting averaged learning curve (mean ± std) across folds")
print("==============================")

train_curves = []
val_curves = []

for i, h in enumerate(fold_histories, start=1):
    if "loss" in h and "val_loss" in h:
        train_curves.append(np.array(h["loss"], dtype=float))
        val_curves.append(np.array(h["val_loss"], dtype=float))
    else:
        print(f"[Warn] Fold {i} missing loss/val_loss keys. Keys: {list(h.keys())}")

if len(train_curves) == 0:
    raise RuntimeError("No usable fold histories found. Ensure your model logs 'loss' and 'val_loss'.")

min_len = min(len(c) for c in train_curves + val_curves)
print(f"[Align] Using first {min_len} epochs across all folds (due to early stopping).")

train_mat = np.stack([c[:min_len] for c in train_curves], axis=0)
val_mat   = np.stack([c[:min_len] for c in val_curves], axis=0)

epochs_axis = np.arange(1, min_len + 1)

train_mean = train_mat.mean(axis=0)
train_std  = train_mat.std(axis=0)

val_mean = val_mat.mean(axis=0)
val_std  = val_mat.std(axis=0)

plt.figure()
plt.plot(epochs_axis, train_mean, label="Train (mean)")
plt.fill_between(epochs_axis, train_mean - train_std, train_mean + train_std, alpha=0.2)

plt.plot(epochs_axis, val_mean, label="Validation (mean)")
plt.fill_between(epochs_axis, val_mean - val_std, val_mean + val_std, alpha=0.2)

plt.title("Cross-Validation Learning Curve (mean ± std)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

- Main script

In [ ]:
# ==============================================================================
# Execution of Data Prep
# ==============================================================================
print("--- Loading, Splitting, and Scaling Data ---")
# Unpack the 3 sets: Train, Validation, and Test (Holdout)
(X_train_raw, y_train_raw, meta_train_raw, 
 X_val_raw, y_val_raw, meta_val_raw, 
 X_test_final, y_test_final, meta_test_final, n_features) = prepare_and_scale_data(filepath, timesteps, holdout_path)

print("\n--- Balancing Labels (Train & Validation) ---")
# Balance Train and Validation independently to prevent leakage
    # We do not balance the final Test set (X_test_final) as it should represent real-world distribution
x_train, y_train, meta_train = balance_single_dataset(X_train_raw, y_train_raw, meta_train_raw, "Train")
x_val, y_val, meta_val = balance_single_dataset(X_val_raw, y_val_raw, meta_val_raw, "Validation")

# Assign Plot Groups of the test set, so we know which is true healhty, severe and labeled when plotting later
X_test_final, y_test_final, meta_test_final = assign_test_set_groups(
    X_test_final, y_test_final, meta_test_final, "Test"
)

print("\n--- Final Dataset Shapes ---")
print(f"Train set:      {x_train.shape}")
print(f"Validation set: {x_val.shape}")
print(f"Test set: {X_test_final.shape} (Saved to disk, untouched)")

print("\n--- UNIQUE MICE DISTRIBUTION ---")
print_unique_mice_per_split(y_train, meta_train, "Train")
print_unique_mice_per_split(y_val, meta_val, "Validation")
print_unique_mice_per_split(y_test_final, meta_test_final, "Test (Holdout)")
print("\n")

# Check Distributions
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_val, counts_val = np.unique(y_val, return_counts=True)
print(f"Train label distribution:      {dict(zip(unique_train, counts_train))}")
print(f"Validation label distribution: {dict(zip(unique_val, counts_val))}")

In [ ]:
# ==============================================================================
# Model Compilation & Training
# ==============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Compile the model
encoder = build_encoder(timesteps, n_features, latent_dim)
decoder = build_decoder(timesteps, n_features, latent_dim, num_classes) 
classifier = build_classifier(timesteps, latent_dim, num_classes)

ss_vae = SemiSupervised_GRU_CVAE(
    encoder, decoder, classifier, 
    alpha=alpha_classification_weight,
    num_classes=num_classes
)
ss_vae.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate))

# --- Define Callbacks ---
# 1. Early Stopping: Stop if no improvement after 20 epochs
early_stopping = EarlyStopping(
    monitor='val_loss',       # Watch the total validation loss
    patience=50,              # Stop after 20 epochs of no improvement
    restore_best_weights=True,# Go back to the best model found, not the last one
    verbose=1
)

# 2. Reduce Learning Rate: If stuck for 10 epochs, lower the LR
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',       # Watch the total validation loss
    factor=0.5,               # Cut learning rate in half (multiply by 0.5)
    patience=10,              # Wait 10 epochs before reducing
    min_lr=1e-7,              # Don't go below this learning rate
    verbose=1
)

# Train
print("\n--- Starting Semi-Supervised Conditional Training ---")
history = ss_vae.fit(
    x_train, y_train,
    epochs=epochs,           
    batch_size=batch_size,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping, reduce_lr] 
)
print("--- Training done ---")

# Save history
np.save(output_dir + 'history_semisupervised.npy', history.history)

- Save the model

In [ ]:
# each part of the model is saved in .keras format
encoder.save(os.path.join(output_dir, f'gru_encoder.keras'))
decoder.save(os.path.join(output_dir, f'gru_decoder.keras'))
classifier.save(os.path.join(output_dir, f'gru_classifier.keras'))
#print("Models saved")

- Plot train/test curves

In [ ]:
# load the history data from the model
history_data = np.load('/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/history_semisupervised.npy', allow_pickle=True).item()

loss = history_data['loss']
val_loss = history_data['val_loss']
recon_loss = history_data['recon_loss']
val_recon_loss = history_data['val_recon']
kl_loss = history_data['kl_loss']
val_kl_loss = history_data['val_kl']

class_loss = history_data['class_loss']
accuracy = history_data['accuracy']
val_class = history_data['val_class']
val_acc = history_data['val_acc']

# figur with 3 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

# --- Plot 1: Accuracy ---
ax1.plot(accuracy, label='Train Accuracy')
ax1.plot(val_acc, label='Validation Accuracy')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

# --- Plot 2: Classification Loss ---
ax2.plot(class_loss, label='Classification Loss')
ax2.plot(val_class, label='Validation Classification Loss')
ax2.set_title('Classification Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

# show the plots
plt.tight_layout()
plt.show()


# figur with 3 subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

# --- Plot 1: Total Loss ---
ax1.plot(loss, label='Train Total Loss')
ax1.plot(val_loss, label='Validation Total Loss')
ax1.set_title('Total Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# --- Plot 2: Reconstruction Loss ---
ax2.plot(recon_loss, label='Train Reconstruction Loss')
ax2.plot(val_recon_loss, label='Validation Reconstruction Loss')
ax2.set_title('Reconstruction Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

# --- Plot 3: KL Divergence Loss ---
ax3.plot(kl_loss, label='Train KL Loss')
ax3.plot(val_kl_loss, label='Validation KL Loss')
ax3.set_title('KL Divergence Loss')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Loss')
ax3.legend()

# show the plots
plt.tight_layout()
plt.show()

print(f"Learning rate: {learning_rate}, Alpha: {alpha_classification_weight}")

In [ ]:
""" plt.figure(figsize=(10, 5))
plt.plot(loss, label="Train Total Loss")
plt.plot(val_loss, '--', label="Validation Total Loss")
plt.title("Total Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig("/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/learning_curve_total_loss.png", dpi=300)
plt.show()

# --- Plot: Accuracy ---
plt.figure(figsize=(10, 5))
plt.plot(accuracy, label='Train Accuracy')
plt.plot(val_acc, '--', label='Validation Accuracy')
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig("/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/learning_curve_accuracy.png", dpi=300)
plt.show()

# --- Plot: Classification Loss ---
plt.figure(figsize=(10, 5))
plt.plot(class_loss, label='Classification Loss')
plt.plot(val_class, '--', label='Validation Classification Loss')
plt.title("Classification Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig("/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/learning_curve_classification.png", dpi=300)
plt.show()

# --- Plot: Reconstruction Loss ---
plt.figure(figsize=(10, 5))
plt.plot(recon_loss, label="Train Reconstruction Loss")
plt.plot(val_recon_loss, '--', label="Validation Reconstruction Loss")
plt.title("Reconstruction Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig("/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/learning_curve_reconstruction.png", dpi=300)
plt.show()

# --- Plot: KL Divergence Loss ---
plt.figure(figsize=(10, 5))
plt.plot(kl_loss, label="Train KL Loss")
plt.plot(val_kl_loss, '--', label="Validation KL Loss")
plt.title("KL Divergence Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig("/Users/asm/Desktop/UNI/MasterThesis/final_results_gru_nonscaled/learning_curve_KL.png", dpi=300)
plt.show()
 """

- UMAP-plotting is done in a separate script as the *umap packages* is unfittable with the *tensorflow*
    - So, save the variables needed for UMAP:

In [ ]:
# --- Exporting All Data For UMAP ---
print("--- Encoding and Saving All Data for UMAP ---")

# 1. Recombine the data using the NEW variable names
# Order: Train -> Validation -> Final Test
X_all = np.concatenate([x_train, x_val, X_test_final], axis=0)
y_all = np.concatenate([y_train, y_val, y_test_final], axis=0)
meta_all = pd.concat([meta_train, meta_val, meta_test_final], axis=0).reset_index(drop=True)

# 2. Generate Latent Sequences (Z) using the trained encoder
    # Here the trained encoder process the test set and predicts where in the learned space the test set will be placed
z_all, z_mean_all, z_log_var_all = encoder.predict(X_all, batch_size=batch_size, verbose=1)

# 3. Save to Disk
# We save z_mean_all because that is what we usually plot (the mean of the latent distribution)
np.save(os.path.join(output_dir, "z_mean_all_semisupervised.npy"), z_mean_all)
np.save(os.path.join(output_dir, "labels_all_semisupervised.npy"), y_all)
meta_all.to_pickle(os.path.join(output_dir, "metadata_semisupervised.pkl"))

print(f"\nSaved combined Train ({len(x_train)}), Validation ({len(x_val)}), and Test ({len(X_test_final)}) data.")
print(f"Total shape: {z_mean_all.shape}")
print(f"Saved to: {output_dir}")

We save the holdout as well, to see how the encoder place the hold-out set in the space

### Evaluation of the hold-out-set

The matrix can only be one on labeled data, so we calculate the confusion matrix for H and S mice

In [ ]:
# ==============================================================================
# CELL 7: Final Test Set Evaluation (Confusion Matrix & Metrics)
# ==============================================================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np

print("\n--- Evaluating Model on Unseen Final Test Set ---")

# 1. Filter out unlabelled data (-1) if any exist in the test set
# (Note: In your current setup, the test set likely only has 0 and 1, but this is safe)
labeled_indices = np.where(y_test_final != -1)[0]
X_test_labeled = X_test_final[labeled_indices]
y_test_true = y_test_final[labeled_indices]

if len(y_test_true) == 0:
    print("No labeled data in the test set to evaluate!")
else:
    # 2. Get latent vectors from the Encoder
    # The encoder outputs [z, z_mean, z_log_var] -> we only need z (index 0)
    z_test, _, _ = encoder.predict(X_test_labeled, verbose=0)
    
    # 3. Get prediction probabilities from the Classifier
    # The classifier takes 'z' and outputs softmax probabilities
    y_test_pred_probs = classifier.predict(z_test, verbose=0)
    
    # 4. Convert probabilities to definitive class labels (0 or 1)
    y_test_pred = np.argmax(y_test_pred_probs, axis=1)
    
    # 5. Generate and Plot the Confusion Matrix
    cm = confusion_matrix(y_test_true, y_test_pred, labels=[0, 1])
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy (0)', 'Severe (1)'])
    
    # Plotting with binary colormap (grayscale)
    disp.plot(cmap=plt.cm.binary, ax=ax, values_format='d', colorbar=False)
    
    # Add a custom colorbar
    cbar = fig.colorbar(disp.im_, ax=ax, shrink=0.6)
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
    
    # 6. Generate and Print the Classification Report
    print("\n--- Classification Report ---")
    report = classification_report(
        y_test_true, 
        y_test_pred, 
        labels=[0, 1], 
        target_names=['Healthy (0)', 'Severe (1)']
    )
    print(report)